# Lista 02 - CKP8366 - TÓPICOS ESPECIAIS EM LÓGICA I

Realizado por Gustavo Fernandez Vidal Vazquez, matrícula 603150. 

# Imports e Auxiliares

In [1]:
from pysat.formula import *
import random
import math
import time
import signal


#### Criação de Fórmula Aleatória

In [2]:
def chooseRandomAtom(k) :
    return (Atom('p' + str(random.randint(1, k))))

operators = ['AND', 'OR', 'NEG']

def formRand(n, k = None):
    if k is None:
        k = n
        
    if (n == 1): 
        return chooseRandomAtom(k)
        
    if (n == 2):
        return Neg(formRand(n-1, k))

    op = random.choice(operators)
    if (op == 'NEG'):
        return Neg(formRand(n-1, k))

    left_size = random.randint(1, n - 2)
    right_size = (n - 1) - left_size
    
    if (op == 'AND'):
        return And(formRand(left_size, k), formRand(right_size, k))             
    if (op == 'OR'):
        return Or(formRand(left_size, k), formRand(right_size, k))    



#### Converter para FNC

In [3]:
def fncon(f):
    mapping = {}
    clauses = []
    counter = 0

    def new_atom():
        nonlocal counter
        counter += 1
        return Atom(("t_" + str(counter)))

    def encode(f):
        if f in mapping:
            return mapping[f]
            
        if isinstance(f, Atom):
            return f

        z = new_atom()
        mapping[f] = z
        
        if isinstance(f, And):
            children = [encode(sub) for sub in f.subformulas]
            for child in children:
                clauses.append(Or(Neg(z), child))
            clauses.append(Or(*([Neg(a) for a in children] + [z])))

        elif isinstance(f, Or):
            children = [encode(sub) for sub in f.subformulas]
            for child in children:
                clauses.append(Or(Neg(child), z))
            clauses.append(Or(Neg(z), *children))
            
        elif isinstance(f, Neg):
            a = encode(f.subformula)
            clauses.append(Or(Neg(z), Neg(a)))
            clauses.append(Or(z, a))
        return z
        
    root = encode(f)
    clauses.append(root)

    return And(*clauses)

#### Representação em Lista de Inteiros

In [4]:
def clausulas(f):
    idp = IDPool()
    clauses = []

    def lit_to_int(lit):
        if isinstance(lit, Atom):
            return idp.obj2id[lit]
        elif isinstance(lit, Neg):
            return -idp.obj2id[lit.subformula]

    if isinstance(f, And):
        for clause in f.subformulas:
            if isinstance(clause, Or):
                c = []
                for lit in clause.subformulas:
                    c.append(lit_to_int(lit))
                clauses.append(c)
            else:
                clauses.append([lit_to_int(clause)])
    else:
        if isinstance(f, Or):
            c = [lit_to_int(l) for l in f.subformulas]
            clauses.append(c)
        else:
            clauses.append([lit_to_int(f)])

    return clauses

# 1. Escreva um programa que usa a biblioteca pysat para construir a fórmula que descreve as condições dos problemas das n-rainhas.

In [5]:
def n_queens_formula(n):
    lista = []

    # 1. Pelo menos uma rainha em cada linha
    for i in range(n):
        clausula = Or(*[Atom(f"p{i},{j}") for j in range(n)])
        lista.append(clausula)

    # 2. No máximo uma rainha por linha
    for i in range(n):
        for j in range(n):
            for k in range(j + 1, n):
                lista.append(
                    Neg(
                        And(
                            Atom(f"p{i},{j}"),
                            Atom(f"p{i},{k}")
                        )
                    )
                )

    # 3. No máximo uma rainha por coluna
    for j in range(n):
        for i in range(n):
            for k in range(i + 1, n):
                lista.append(
                    Neg(
                        And(
                            Atom(f"p{i},{j}"),
                            Atom(f"p{k},{j}")
                        )
                    )
                )

    # 4. Diagonal principal (\)
    for i in range(n):
        for j in range(n):
            for k in range(1, n):
                if i + k < n and j + k < n:
                    lista.append(
                        Neg(
                            And(
                                Atom(f"p{i},{j}"),
                                Atom(f"p{i+k},{j+k}")
                            )
                        )
                    )

    # 5. Diagonal secundária (/)
    for i in range(n):
        for j in range(n):
            for k in range(1, n):
                if i + k < n and j - k >= 0:
                    lista.append(
                        Neg(
                            And(
                                Atom(f"p{i},{j}"),
                                Atom(f"p{i+k},{j-k}")
                            )
                        )
                    )

    # Fórmula final
    return And(*lista)

# 2. Utilize a pysat para implementar os seguintes estratégias de resolução:

In [6]:
# Recebe duas cláusulas
# Para cada literal em c1
#   Se -literal estiver em c2
#       conseguimos gerar c1 U c2 - ({lit} U - {lit})
def resolve(c1, c2):
    resolvents = []

    for lit in c1:
        if -lit in c2:
            new_clause = []

            # c1 sem lit
            for x in c1:
                if x != lit:
                    new_clause.append(x)

            # c2 sem -lit (sem duplicar)
            for x in c2:
                if x != -lit and x not in new_clause:
                    new_clause.append(x)

            taut = False
            # Tratamento para não incluir tautologias 
            for x in new_clause:
                if -x in new_clause:
                    taut = True
                    break

            if not taut:
                resolvents.append(new_clause)

    return resolvents

## i) Resolução Ingênua

In [7]:
def resolution_naive(clauses):
    new = []

    while True:
        n = len(clauses)

        # todos os pares
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve(clauses[i], clauses[j])

                for r in resolvents:
                    if r == []:
                        return True  # UNSAT 
                    if r not in clauses and r not in new:
                        new.append(r)

        if not new:
            return False  # SAT 

        clauses.extend(new)
        new = []

## ii) Resolução Linear

In [8]:
def resolution_linear(clauses):
    # tenta começar com cada cláusula
    for start in clauses:
        stack = [(start, clauses)]

        while stack:
            current, usable = stack.pop()

            for c in usable:
                if c == current:
                    continue

                resolvents = resolve(current, c)

                for r in resolvents:
                    if r == []:
                        return True  # UNSAT

                    stack.append((r, usable))

    return False

## iii) Resolução Direcional

In [9]:
def get_vars(clauses):
    return sorted(set(abs(lit) for c in clauses for lit in c))


def resolution_directed(clauses, var_order=None):
    if var_order is None:
        var_order = get_vars(clauses)

    buckets = {v: [] for v in var_order}

    for c in clauses:
        if not c:
            return True  

        smallest_var = min(abs(x) for x in c)
        buckets[smallest_var].append(c)

    for var in var_order:
        bucket = buckets[var]

        pos = []
        neg = []

        for c in bucket:
            if var in c:
                pos.append(c)
            elif -var in c:
                neg.append(c)

        for c1 in pos:
            for c2 in neg:
                resolvents = resolve(c1, c2)

                for r in resolvents:
                    if r == []:
                        return True  # UNSAT

                    if r:  
                        smallest_var = min(abs(x) for x in r)

                        if r not in buckets[smallest_var]:
                            buckets[smallest_var].append(r)

    return False  # SAT

# 3. Benchmark (Timeout de 10s por exemplo) por conta da viabilidade da execução

In [10]:

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

signal.signal(signal.SIGALRM, timeout_handler)


def run_with_timeout(func, args, timeout=10):
    try:
        signal.alarm(timeout)
        start = time.time()
        result = func(*args)
        end = time.time()
        signal.alarm(0)
        return ("ok", end - start, result)
    except TimeoutException:
        return ("timeout", timeout, None)


# k entre √n e 2√n
def choose_k(n):
    low = int(math.sqrt(n))
    high = int(2 * math.sqrt(n))
    return random.randint(low, max(low, high))


def generate_clauses(n):
    k = choose_k(n)
    formula = formRand(n, k)
    cnf = fncon(formula)
    clauses = clausulas(cnf)
    return clauses


def benchmark_algorithm(alg_func, name):
    ns = [10,20,30,40,50,60,70,80,90,100]

    print(f"\n=== {name} ===")

    for n in ns:
        times = []
        timeouts = 0

        for _ in range(2):
            clauses = generate_clauses(n)

            status, t, _ = run_with_timeout(alg_func, (clauses,), timeout=2)

            if status == "ok":
                times.append(t)
            else:
                timeouts += 1

        avg = sum(times)/len(times) if times else 0

        print(f"n={n} | avg={avg:.4f}s | timeouts={timeouts}")


def run_benchmarks():
    benchmark_algorithm(resolution_directed, "Directed")
    benchmark_algorithm(resolution_linear, "Linear")
    benchmark_algorithm(resolution_naive, "Naive")



In [11]:
run_benchmarks()


=== Directed ===
n=10 | avg=0.0000s | timeouts=0
n=20 | avg=0.0002s | timeouts=0
n=30 | avg=0.0003s | timeouts=0
n=40 | avg=0.0004s | timeouts=0
n=50 | avg=0.0004s | timeouts=0
n=60 | avg=0.0282s | timeouts=0
n=70 | avg=0.5433s | timeouts=1
n=80 | avg=0.0543s | timeouts=1
n=90 | avg=0.0000s | timeouts=2
n=100 | avg=0.0111s | timeouts=1

=== Linear ===
n=10 | avg=0.0000s | timeouts=2
n=20 | avg=0.0000s | timeouts=2
n=30 | avg=0.0000s | timeouts=2
n=40 | avg=0.0000s | timeouts=2
n=50 | avg=0.0000s | timeouts=2
n=60 | avg=0.0000s | timeouts=2
n=70 | avg=0.0000s | timeouts=2
n=80 | avg=0.0000s | timeouts=2
n=90 | avg=0.0000s | timeouts=2
n=100 | avg=0.0000s | timeouts=2

=== Naive ===
n=10 | avg=0.0000s | timeouts=2
n=20 | avg=0.0000s | timeouts=2
n=30 | avg=0.0000s | timeouts=2
n=40 | avg=0.0000s | timeouts=2
n=50 | avg=0.0000s | timeouts=2
n=60 | avg=0.0000s | timeouts=2
n=70 | avg=0.0000s | timeouts=2
n=80 | avg=0.0000s | timeouts=2
n=90 | avg=0.0000s | timeouts=2
n=100 | avg=0.0000s |

# 4. Benchmark da direcionada

In [12]:
def benchmark_q4():
    n = 100

    print("\n=== Q4: Directed com 10 ordens aleatórias (n=100) ===", flush=True)

    clauses = generate_clauses(n)

    vars_list = get_vars(clauses)

    times = []
    timeouts = 0

    for i in range(10):
        order = vars_list[:]
        random.shuffle(order)

        print(f"run {i+1}/10", flush=True)

        status, t, _ = run_with_timeout(
            resolution_directed,
            (clauses, order),
            timeout=2
        )

        if status == "ok":
            times.append(t)
            print(f"  time={t:.4f}s", flush=True)
        else:
            timeouts += 1
            print("  timeout", flush=True)

    avg = sum(times)/len(times) if times else 0

    print(f"\navg={avg:.4f}s | timeouts={timeouts}", flush=True)

In [13]:
benchmark_q4()


=== Q4: Directed com 10 ordens aleatórias (n=100) ===
run 1/10
  time=0.0003s
run 2/10
  time=0.0003s
run 3/10
  time=0.0003s
run 4/10
  time=0.0003s
run 5/10
  time=0.0003s
run 6/10
  time=0.0002s
run 7/10
  time=0.0002s
run 8/10
  time=0.0003s
run 9/10
  time=0.0003s
run 10/10
  time=0.0003s

avg=0.0003s | timeouts=0
